In [1]:
from collections import namedtuple

from docplex.mp.model import Model
from docplex.util.environment import get_environment
import pandas as pd
import numpy as np

# Reading in Data

In [2]:
newStudentInfo = pd.read_excel("Interview small data.xlsx", sheet_name=0)
teacherInfo = pd.read_excel("Interview small data.xlsx", sheet_name=1)
existingStudentInfo = pd.read_excel("Interview small data.xlsx", sheet_name=2)


# Data Pre-processing

Remap tutoringNeeds column to numerical equivalent (Extensive : 2, Normal : 1) to facilitate contraint creation below. Created dictonaries mapping studentIds to their numerical tutoringNeeds and tuitionCenters to faciliate contraint and objective function creation respectively

In [3]:
# Processing newStudents
newStudentInfo["tutoringNeed_num"] = newStudentInfo["tutoringNeed"].map({"Extensive" : 2, "Normal" : 1})
newStudentNeeds = dict(zip(newStudentInfo["studentId"], newStudentInfo["tutoringNeed_num"]))
newStudentCentres = dict(zip(newStudentInfo["studentId"], newStudentInfo["tuitionCentre"]))

Similar to newStudents, with an additional dictionary mapping tutorIds to their maximum capacity. 

In [4]:
# Processing teacherInfo
teacherInfo["tutoringSkills_num"] = teacherInfo["tutoringSkills"].map({"Extensive" : 2, "Normal" : 1})
teacherPrefCen = dict(zip(teacherInfo["tutorId"],tuple(zip(teacherInfo["preferredCentre1"],teacherInfo["preferredCentre2"]))))
teacherSkills = dict(zip(teacherInfo["tutorId"],teacherInfo["tutoringSkills_num"]))
teacherMaxCap = dict(zip(teacherInfo["tutorId"], teacherInfo["maxOverallCapacity"]))

Since existing students with and active status of False no longer need tuition, I filter them out. The existing students after that already have a tutor, so I just create 1 dictionary to reflect those pairings

In [6]:
# Processing existingStudentInfo
existingStudentInfo = existingStudentInfo[existingStudentInfo["active"]]
existingStudents = dict(zip(existingStudentInfo["studentId"], existingStudentInfo["tutorId"]))

In [7]:
students = list(newStudentNeeds.keys()) + list(existingStudents.keys())
teachers = list(teacherSkills.keys())

$x_{ij}$ is a binary decision variable, where 1 indicates that student i is being taught by tutor j, and 0 otherwise

In [8]:
mdl = Model("Tutor Assignment Problem")
x = mdl.binary_var_matrix(students, teachers, name = "Assignment")

# Constraints

Adding student-tutor pairs for existing students as constraints. 

In [9]:
for student in existingStudents.keys():
    for teacher in teachers:
        if teacher == existingStudents[student]:
            mdl.add_constraint(x[student, teacher] == 1, ctname = "Existing Students_teacher")
        else:
            mdl.add_constraint(x[student, teacher] == 0, ctname = "Existing Students_not_teacher")

Adding the maximum 1 tutor per student constraint for new students. Previous code snippet makes this unecessary for existing students.

In [10]:
for student in newStudentCentres.keys():
    mdl.add_constraint(mdl.sum(x[student, teacher] for teacher in teachers) == 1, ctname = "Max Num of tutors per student")

Adding constraints preventing Normal skilled Tutors from being paired with Extensive needs new students. Once again, since existing students already have valid pairings, these constraints have not been created for them

In [11]:
for student in newStudentCentres.keys():
    for teacher in teachers:
        if teacherSkills[teacher] < newStudentNeeds[student]:
            # This teacher is under-qualified for this specific student
            mdl.add_constraint(x[student, teacher] == 0, ctname=f"forbidden_{student}_{teacher}")

Adding constraint preventing tutors from paired with more students then their maxcap allows.

In [12]:
for teacher in teachers:
    mdl.add_constraint(mdl.sum(x[student, teacher] for student in newStudentNeeds.keys()) <= teacherMaxCap[teacher], ctname = "Max Num of students per teachers less than teacher max cap")

# Question A

Creating a total satisfaction variable, measuring how "satisfied" tutors are with their students depending on their centre preferences and the students' centres. 

In [17]:
def get_satisfaction(s,t):
    try: 
        return 2 - teacherPrefCen[t].index(newStudentCentres[s])
    except:
        return 0
total_satisfaction = mdl.sum(mdl.sum(x[i,j] * get_satisfaction(i, j) for i in students) for j in teachers)
total_satisfaction

docplex.mp.LinearExpr(2Assignment_S0001_A001+Assignment_S0001_A002+Assignment_S0001_A004+2Assignment_S0001_A006+2Assignment_S0001_A007+2Assignment_S0002_A003+Assignment_S0002_A008+2Assignment_S0002_A010+Assignment_S0003_A005+Assignment_S0003_A006+2Assignment_S0003_A008+2Assignment_S0003_A009+2Assignment_S0004_A001+Assignment_S0004_A002+Assignment_S0004_A004+2Assignment_S0004_A006+2Assignment_S0004_A007+Assignment_S0005_A005+Assignment_S0005_A006+2Assignment_S0005_A008+2Assignment_S0005_A009+Assignment_S0006_A001+2Assignment_S0006_A002+Assignment_S0006_A003+2Assignment_S0006_A004+2Assignment_S0006_A005+Assignment_S0006_A007+Assignment_S0006_A009+Assignment_S0006_A010+2Assignment_S0007_A001+Assignment_S0007_A002+Assignment_S0007_A004+2Assignment_S0007_A006+2Assignment_S0007_A007+Assignment_S0008_A005+Assignment_S0008_A006+2Assignment_S0008_A008+2Assignment_S0008_A009+Assignment_S0009_A001+2Assignment_S0009_A002+Assignment_S0009_A003+2Assignment_S0009_A004+2Assignment_S0009_A005+Assignmen

total_teachers refers to the total number of teachers teaching at least 1 student

In [23]:
total_teachers = mdl.sum(mdl.sum(x[i,j] for i in students) >= 1 for j in teachers)

Since this is a multi-objective problem(minimizing total_teachers while maximizing total_satisfaction), the objective function containes both variables. Maximizing total_satisfaction is equivalent to minimizing (-total_satisfaction). From the question, it seems that greater weight should be placed on minimizing total_teachers instead of (-total_satisfaction), therefore it has a higher weight.

In [24]:
mdl.minimize(0.7 * total_teachers - 0.3 * total_satisfaction) 

# Solve
sol = mdl.solve()

if sol:
    print(sol)
else:
    print("No solution found.")

solution for: Tutor Assignment Problem
objective: -7.9
status: OPTIMAL_SOLUTION(2)
Assignment_S0001_A007=1
Assignment_S0002_A003=1
Assignment_S0003_A005=1
Assignment_S0004_A007=1
Assignment_S0005_A009=1
Assignment_S0006_A009=1
Assignment_S0007_A007=1
Assignment_S0008_A009=1
Assignment_S0009_A005=1
Assignment_S0010_A003=1
Assignment_S0011_A009=1
Assignment_S0012_A003=1
Assignment_S0013_A003=1
Assignment_S0014_A005=1
Assignment_S0015_A007=1
Assignment_S0016_A005=1
Assignment_S0017_A005=1
Assignment_S0018_A010=1
Assignment_S0019_A003=1
Assignment_S0020_A005=1
Assignment_S0036_A003=1
Assignment_S0057_A005=1
Assignment_S0059_A005=1
Assignment_S0061_A005=1
Assignment_S0062_A005=1
Assignment_S0086_A007=1
Assignment_S0088_A007=1
Assignment_S0091_A007=1
Assignment_S0092_A007=1
Assignment_S0093_A007=1
Assignment_S0110_A010=1
Assignment_S0119_A010=1



# Question B

In an ideal scenario, the workload of all the teachers would be balanced if they all taught the same number of students. Since each student can only have 1 teacher, this ideal number is equal to the average number of student per teacher in the problem. Then to maximize the "balance" in the pairings, we can minimize the imbalance, which would be the absolute difference between the number of student a tutor is teaching from the ideal number. 

In [34]:
avg_students = len(students)/len(teachers)
imbalance = mdl.sum(mdl.abs(mdl.sum(x[i,j] for i in students) - avg_students) for j in teachers)

As in the previous question, though we want to maximize total_satisfaction (minimize -total_satisfaction), we are more concerned with minimzing the imbalance. This is reflected in their respective weights.

In [35]:
mdl.minimize(0.7 * imbalance - 0.3 * total_satisfaction) 

# Solve
sol = mdl.solve()

if sol:
    print(sol)
else:
    print("No solution found.")

solution for: Tutor Assignment Problem
objective: -6.86
status: OPTIMAL_SOLUTION(2)
Assignment_S0001_A006=1
Assignment_S0002_A010=1
Assignment_S0003_A009=1
Assignment_S0004_A001=1
Assignment_S0005_A008=1
Assignment_S0005_A009=-0
Assignment_S0006_A001=1
Assignment_S0007_A006=1
Assignment_S0008_A009=1
Assignment_S0009_A002=1
Assignment_S0010_A003=1
Assignment_S0010_A010=-0
Assignment_S0011_A006=-0
Assignment_S0011_A009=1
Assignment_S0012_A002=1
Assignment_S0013_A008=1
Assignment_S0014_A004=1
Assignment_S0015_A006=1
Assignment_S0016_A002=1
Assignment_S0017_A004=1
Assignment_S0018_A008=1
Assignment_S0018_A010=0
Assignment_S0019_A003=1
Assignment_S0020_A004=1
Assignment_S0036_A003=1
Assignment_S0057_A005=1
Assignment_S0059_A005=1
Assignment_S0061_A005=1
Assignment_S0062_A005=1
Assignment_S0086_A007=1
Assignment_S0088_A007=1
Assignment_S0091_A007=1
Assignment_S0092_A007=1
Assignment_S0093_A007=1
Assignment_S0110_A010=1
Assignment_S0119_A010=1



In [26]:
existingStudentInfo["tutorId"].unique()

<StringArray>
['A003', 'A005', 'A007', 'A010']
Length: 4, dtype: str

In [21]:
newStudentInfo[newStudentInfo["studentId"].isin(["S0001","S0004","S0006","S0007","S0015"])]

,studentId,tutoringNeed,tuitionCentre,tutoringNeed_num
0,S0001,Normal,East,1
3,S0004,Normal,East,1
5,S0006,Extensive,North,2
6,S0007,Normal,East,1
14,S0015,Normal,East,1


In [ ]:
# total_tutors = mdl.sum(y[j] for j in tutors)
# total_satisfaction = mdl.sum(x[i,j] * get_satisfaction(i, j) for i in students for j in tutors)

# # We want to minimize tutors and maximize satisfaction (by subtracting it)
# # Weighting: 10 to 1 ratio to prioritize minimizing teachers
# mdl.minimize(10 * total_tutors - 1 * total_satisfaction)

In [63]:
print(mdl.export_as_lp_string())

\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: Tutor Assignment Problem

Minimize
 obj:
Subject To
 _Existing_Students_not_teacher: Assignment_S0036_A001 = 0
 _Existing_Students_not_teacher#1: Assignment_S0036_A002 = 0
 _Existing_Students_teacher: Assignment_S0036_A003 = 1
 _Existing_Students_not_teacher#3: Assignment_S0036_A004 = 0
 _Existing_Students_not_teacher#4: Assignment_S0036_A005 = 0
 _Existing_Students_not_teacher#5: Assignment_S0036_A006 = 0
 _Existing_Students_not_teacher#6: Assignment_S0036_A007 = 0
 _Existing_Students_not_teacher#7: Assignment_S0036_A008 = 0
 _Existing_Students_not_teacher#8: Assignment_S0036_A009 = 0
 _Existing_Students_not_teacher#9: Assignment_S0036_A010 = 0
 _Existing_Students_not_teacher#10: Assignment_S0057_A001 = 0
 _Existing_Students_not_teacher#11: Assignment_S0057_A002 = 0
 _Existing_Students_not_teacher#12: Assignment_S0057_A003 = 0
 _Existing_Students_not_teacher#13: Assignment_S0057_A004 = 0
 _Existing_Students